In [0]:
#importing necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, roc_curve

#Data Loading and Preprocessing
file_path ="/Workspace/Users/mphoshomalivhonani@gmail.com/PYTHON/BrightLearn_Python/Patient Readmission/Expanded_Patient_Readmission_Data_(1).csv"
df = pd.read_csv(file_path)

df.head()

#Data Processing

In [0]:
# Encoding categorical variables
le = LabelEncoder()
df['Gender'] = le.fit_transform(df['Gender'])
df['Admission Type'] = le.fit_transform(df['Admission Type'])
df['Readmission'] = le.fit_transform(df['Readmission'])

# Feature Selection (all features except Patient ID and Readmission)
X = df.drop(['Patient ID', 'Readmission'], axis=1)
y = df['Readmission']

# Feature Scaling (Crucial for distance-based algorithms like KNN)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

df.head()

In [0]:
print("Preprocessing Complete.")

#Model Implementation

In [0]:
from xgboost import XGBClassifier

In [0]:
models = {
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=5),
    'XGBoost': XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
}

results = []
for name, model in models.items():
    # Train the model
    model.fit(X_train, y_train)
    
    # Make predictions
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else None
    
    # Evaluate the model
    metrics = {
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall': recall_score(y_test, y_pred, zero_division=0),
        'F1 Score': f1_score(y_test, y_pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_test, y_prob) if y_prob is not None else 'N/A'
    }
    results.append(metrics)

    print()

In [0]:
X = df.drop(['Patient ID', 'Readmission'], axis=1)
y = df['Readmission']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [0]:
# Random Forest Model
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

In [0]:
# Predictions
y_pred = rf_model.predict(X_test)
y_prob = rf_model.predict_proba(X_test)[:, 1]

#Model Evaluation

In [0]:
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}")

In [0]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

In [0]:
# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f'ROC Curve (AUC = {roc_auc_score(y_test, y_prob):.2f})')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.show()

In [0]:
#Feature Importance
importances = pd.DataFrame({'Feature': X.columns, 'Importance': rf_model.feature_importances_})
importances = importances.sort_values(by='Importance', ascending=False)
print("\nFeature Importances:")
print(importances)

#Comparison and Visualization of Results

In [0]:
# Correlation Heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(df.drop('Patient ID', axis=1).corr(), annot=True, cmap='coolwarm')
plt.title('Feature Correlation Heatmap')
plt.show()

In [0]:
plt.figure(figsize=(8, 5))
sns.countplot(x='Readmission', data=df)
plt.title('Readmission Distribution (0: No, 1: Yes)')
plt.show()

In [0]:
# 3. Comparison Table
results_df = pd.DataFrame(results)
print("--- Model Performance Comparison ---")
print(results_df.to_string(index=False))

# 4. Visualization of Results
plt.figure(figsize=(12, 6))
results_melted = results_df.melt(
    id_vars="Model", value_vars=["Accuracy", "F1 Score", "ROC-AUC"]
)
sns.barplot(x="Model", y="value", hue="variable", data=results_melted)
plt.title("Comparison of Machine Learning Models")
plt.ylim(0, 1)
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.savefig("/Workspace/Users/mphoshomalivhonani@gmail.com/PYTHON/BrightLearn_Python/model_comparison.png")